# Grounded VQG reproduction -- full training on Colab Pro

Reproduces *Automatic Generation of Grounded Visual Questions* (Zhang et al., IJCAI 2017), architecture-faithful, with library substitutions for deprecated/unreachable dependencies:
- Original Torch/Lua DenseCap -> [`soloist97/densecap-pytorch`](https://github.com/soloist97/densecap-pytorch) (only affects the diversity/quality of the *input* captions, not the VQG model architecture -- see this project's README for the full writeup).
- `densecap-pytorch`'s own pretrained checkpoint (OneDrive/BaiduYun) turned out to be unreachable -- broken share link on OneDrive, anonymous-download blocked on BaiduYun -- so this notebook **trains densecap-pytorch itself** on Visual Genome (cell 6), using a lightly patched `train.py` (native `torch.cuda.amp` instead of the unmaintained NVIDIA Apex dependency, plus checkpoint-resume support upstream doesn't have). No change to the model architecture or hyperparameters, just robustness for a multi-hour Colab job.
- Original NeuralTalk2 baseline -> not reproduced here (this notebook trains the paper's actual model only).

**Before running:** Runtime -> Change runtime type -> GPU (A100 recommended if your Colab Pro quota allows; T4 works, just slower).

**This will take hours, possibly across multiple sessions.** COCO images (~26GB) + Visual Genome (~15GB) + DenseCap training (10 epochs over ~108k images) + full VQG training over VQA v1 (764k questions) or Visual7W (327k QA pairs) is a genuinely long pipeline. Everything below writes intermediate artifacts to Google Drive so you can stop and resume -- the DenseCap training step in particular is designed to survive a disconnect and pick back up automatically.

## 0. Config -- fill these in

In [ ]:
GITHUB_REPO_URL = "https://github.com/malimustafaa/Automatic-Generation-of-Grounded-Visual-Questions.git"
DATASET = "vqa"  # "vqa" or "visual7w"
DRIVE_ROOT = "/content/drive/MyDrive/grounded-vqg-reproduction"  # all downloads/checkpoints persist here

# DenseCap-pytorch: we train this ourselves (see cell 6) rather than relying on the
# author's pretrained checkpoint, which turned out to be unreachable via both OneDrive
# (broken share link) and BaiduYun (blocks anonymous downloads over ~1GB). These paths
# are where OUR trained checkpoint/config end up -- nothing to fill in by hand here.
DENSECAP_MODEL_PARAMS_DIR = f"{DRIVE_ROOT}/densecap_pytorch/model_params"  # Drive-backed: survives disconnects
DENSECAP_MODEL_NAME = "train_all_val_all_bz_2_epoch_10_inject_init"
# Prefer the best-validation-mAP checkpoint (what the densecap-pytorch maintainer told
# users to actually use, per github.com/soloist97/densecap-pytorch/issues/2) over the
# final-epoch "_end" one; cell 6d falls back to "_end" only if "_best" was never written
# (possible if training finished before any 20k-iteration eval improved on it).
DENSECAP_CHECKPOINT_BEST = f"{DENSECAP_MODEL_PARAMS_DIR}/{DENSECAP_MODEL_NAME}_best.pth.tar"
DENSECAP_CHECKPOINT_END = f"{DENSECAP_MODEL_PARAMS_DIR}/{DENSECAP_MODEL_NAME}_end.pth.tar"
DENSECAP_CONFIG = f"{DENSECAP_MODEL_PARAMS_DIR}/{DENSECAP_MODEL_NAME}/config.json"

## 1. Mount Drive + clone this repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

# If the repo already exists (runtime didn't actually reset), `git pull` instead of
# `git clone` -- a bare clone silently fails with "destination path already exists"
# in that case, leaving whatever stale script versions were already on disk, which is
# exactly what caused a fix pushed to GitHub to not actually take effect after a
# "rerun all cells" that didn't go through a genuinely fresh runtime.
if os.path.exists("/content/grounded-vqg-reproduction"):
    print("Repo already present -- pulling latest instead of cloning.")
    !cd /content/grounded-vqg-reproduction && git pull
else:
    !git clone {GITHUB_REPO_URL} /content/grounded-vqg-reproduction
%cd /content/grounded-vqg-reproduction

## 2. Install dependencies

In [ ]:
!pip install -q pyyaml pycocoevalcap tqdm
# torch/torchvision/numpy/pillow are already present on Colab images.

# Run the smoke test first -- if this fails, nothing downstream will work either,
# and it takes seconds rather than hours to find out.
!python -m tests.smoke_test

## 3. Download GloVe + COCO images (persisted to Drive)

In [ ]:
!bash scripts/download_glove.sh {DRIVE_ROOT}/data

# COCO images extract to LOCAL disk (/content/coco), not the Drive mount -- reading/
# writing ~123k small files through Drive's FUSE layer is both slow and prone to
# sporadic I/O errors at that scale (confirmed in practice, including an attempted
# Drive->local rsync migration that turned out just as slow: rsync still has to open
# each file individually through the same FUSE mount, so bulk-copying doesn't dodge
# the bottleneck either -- a fresh network download + local unzip is dramatically
# faster than moving data off Drive by any method). If train2014/val2014 already exist
# on Drive from an earlier run, that's now dead weight you can delete whenever
# convenient -- it's not used anywhere in this pipeline.
import os
COCO_LOCAL = "/content/coco"
os.makedirs(COCO_LOCAL, exist_ok=True)
!bash scripts/download_coco_images.sh {COCO_LOCAL} {DRIVE_ROOT}/data/coco_zips

## 4. Download & flatten the question dataset (VQA v1 or Visual7W)

In [ ]:
if DATASET == "vqa":
    !python scripts/prepare_vqa.py --dest_dir {DRIVE_ROOT}/data/vqa_raw --out {DRIVE_ROOT}/data/questions.json
else:
    !python scripts/prepare_visual7w.py --dest_dir {DRIVE_ROOT}/data/visual7w_raw --coco_dir {DRIVE_ROOT}/data/coco --out {DRIVE_ROOT}/data/questions.json

## 5. Extract frozen VGG-16 image features (300-d, paper Sec 3.2/4.4)

In [ ]:
FEATURES_PATH = f"{DRIVE_ROOT}/data/image_features.npz"
!python scripts/extract_image_features.py \
  --questions {DRIVE_ROOT}/data/questions.json \
  --image_root {COCO_LOCAL} \
  --out_path {FEATURES_PATH}

## 6. DenseCap: train it ourselves, then generate candidate captions

The author's pretrained checkpoint turned out to be unreachable (OneDrive: broken share link; BaiduYun: blocks anonymous downloads of this size). So instead: download Visual Genome, preprocess it with `densecap-pytorch`'s own `preprocess.py`, then train with our patched `train.py` (native AMP instead of the unmaintained NVIDIA Apex dependency, plus checkpoint-resume support the upstream script doesn't have -- no changes to model architecture or hyperparameters).

**This step alone can take many hours.** If your Colab session disconnects partway through training, just re-run the training cell (6c) -- it auto-detects the last saved epoch on Drive and continues from there instead of restarting. Everything else in this section (VG download, preprocessing) is safe to re-run too; each step skips work that's already done.

In [ ]:
# 6a. Clone densecap-pytorch and install its (non-Apex) dependencies
import os
if not os.path.exists("/content/densecap-pytorch"):
    !git clone https://github.com/soloist97/densecap-pytorch.git /content/densecap-pytorch
else:
    print("densecap-pytorch already cloned, skipping.")

!pip install -q h5py tensorboard prefetch_generator tqdm
# NOTE: upstream's README also lists NVIDIA Apex as a dependency -- we don't need it,
# our patched train.py (cell 6c) uses PyTorch's native torch.cuda.amp instead.
#
# prefetch_generator (used by their DataLoaderPFG) is missing from the README
# entirely -- there's no requirements.txt in the repo, so this list was built by
# grepping every import across all six files the training path touches, not from
# their docs.

# Patch 1: model/evaluator.py imports `from nlgeval.pycocoevalcap.meteor.meteor
# import Meteor` for METEOR-based validation during training. nlg-eval is NOT on
# PyPI (confirmed) and installing it from GitHub source pulls in Theano + a pinned
# old gensim for ONE class we need, for metrics we don't use. pycocoevalcap
# (already installed in cell 2) vendors the same underlying MS-COCO Meteor code
# (confirmed by diffing both files directly), so we patch the import instead of
# installing nlg-eval. The two versions clean up differently -- nlg-eval's Meteor
# has an explicit close() method, pycocoevalcap's relies on __del__ -- so that
# call needs removing too, or it just trades one crash for another.
evaluator_path = "/content/densecap-pytorch/model/evaluator.py"
with open(evaluator_path) as f:
    content = f.read()
content = content.replace(
    "from nlgeval.pycocoevalcap.meteor.meteor import Meteor",
    "from pycocoevalcap.meteor.meteor import Meteor",
)
content = content.replace(
    "        meteor_scorer.close()\n",
    "        # pycocoevalcap's Meteor cleans up via __del__, no explicit close() method\n",
)
with open(evaluator_path, "w") as f:
    f.write(content)
print("Patched model/evaluator.py to use pycocoevalcap instead of nlg-eval.")

# Patch 2: this codebase targets PyTorch ~1.4 (per its README), where
# pack_padded_sequence's `lengths` argument accepted a CUDA tensor. Modern PyTorch
# requires it on CPU specifically (even though the rest of the batch is legitimately
# on GPU) -- three call sites hit this, one in box_describer.py and two in
# roi_heads.py's caption_loss (which runs immediately after box_describer in the
# training forward pass, so fixing only one would just crash on the next line).
box_describer_path = "/content/densecap-pytorch/model/box_describer.py"
with open(box_describer_path) as f:
    content = f.read()
content = content.replace(
    "        rnn_input_pps = pack_padded_sequence(word_emb, lengths=cap_lens, batch_first=True, enforce_sorted=False)",
    "        rnn_input_pps = pack_padded_sequence(word_emb, lengths=cap_lens.cpu(), batch_first=True, enforce_sorted=False)",
)
with open(box_describer_path, "w") as f:
    f.write(content)

roi_heads_path = "/content/densecap-pytorch/model/roi_heads.py"
with open(roi_heads_path) as f:
    content = f.read()
content = content.replace(
    "    predict_pps = pack_padded_sequence(caption_predicts, caption_length, batch_first=True, enforce_sorted=False)",
    "    predict_pps = pack_padded_sequence(caption_predicts, caption_length.cpu(), batch_first=True, enforce_sorted=False)",
)
content = content.replace(
    "    target_pps = pack_padded_sequence(caption_gt[:, 1:], caption_length, batch_first=True, enforce_sorted=False)",
    "    target_pps = pack_padded_sequence(caption_gt[:, 1:], caption_length.cpu(), batch_first=True, enforce_sorted=False)",
)
with open(roi_heads_path, "w") as f:
    f.write(content)
print("Patched box_describer.py and roi_heads.py: pack_padded_sequence lengths -> CPU.")

In [ ]:
# 6b. Download Visual Genome + preprocess into densecap-pytorch's expected format.
# Extracts locally (not to Drive) for the same reason as the COCO download -- ~108k
# small image files unzipped onto Drive's FUSE mount is dramatically slower than local
# disk. Only the ~15GB zip files themselves are cached on Drive, so re-running this on
# a fresh session doesn't re-download them.
!bash scripts/download_visual_genome.sh /content/visual-genome {DRIVE_ROOT}/data/vg_zips

import os
os.makedirs("/content/densecap-pytorch/data", exist_ok=True)
!ln -sfn /content/visual-genome /content/densecap-pytorch/data/visual-genome

%cd /content/densecap-pytorch
!python preprocess.py \
  --region_data /content/visual-genome/region_descriptions.json \
  --image_data /content/visual-genome/image_data.json \
  --split_json info/densecap_splits.json \
  --pickle_output ./data/VG-regions-dicts-lite.pkl \
  --h5_output ./data/VG-regions-lite.h5
%cd /content/grounded-vqg-reproduction

In [ ]:
# 6c. Train (RESUME-SAFE: if the session disconnects, just re-run this cell --
# it auto-loads the last epoch checkpoint from Drive and continues).
import os
os.environ["DENSECAP_MODEL_PARAMS_DIR"] = DENSECAP_MODEL_PARAMS_DIR

%cd /content/densecap-pytorch
!cp /content/grounded-vqg-reproduction/scripts/densecap_train_patched.py .
!mkdir -p {DENSECAP_MODEL_PARAMS_DIR}
!python densecap_train_patched.py
%cd /content/grounded-vqg-reproduction

In [ ]:
# 6d. Generate candidate captions for every image using our trained checkpoint.
# Prefer the best-val-mAP checkpoint; fall back to the final-epoch one only if "_best"
# was never written (e.g. training completed before any 20k-iteration eval improved on it).
if os.path.exists(DENSECAP_CHECKPOINT_BEST):
    DENSECAP_CHECKPOINT = DENSECAP_CHECKPOINT_BEST
elif os.path.exists(DENSECAP_CHECKPOINT_END):
    print(f"No '_best' checkpoint found, falling back to '_end': {DENSECAP_CHECKPOINT_END}")
    DENSECAP_CHECKPOINT = DENSECAP_CHECKPOINT_END
else:
    raise AssertionError(
        f"Training hasn't finished yet -- neither {DENSECAP_CHECKPOINT_BEST} nor "
        f"{DENSECAP_CHECKPOINT_END} exists. Re-run cell 6c to continue training "
        "(it resumes automatically from the last epoch)."
    )

# Processes BOTH train2014 and val2014 -- VQA v1 references images from both splits,
# and build_manifest.py silently drops any question whose image has no candidates, so
# skipping a split doesn't error, it just quietly loses that data. --result_dir is on
# Drive (not /content) so each split's completed result.json survives a disconnect --
# re-running this cell after a crash on one split reuses the other split's finished
# result.json instead of redoing it (this is what cost ~3hrs once already).
!python scripts/run_densecap.py \
  --densecap_repo /content/densecap-pytorch \
  --config_json {DENSECAP_CONFIG} \
  --checkpoint {DENSECAP_CHECKPOINT} \
  --img_dirs {COCO_LOCAL}/train2014 {COCO_LOCAL}/val2014 \
  --result_dir {DRIVE_ROOT}/data/densecap_raw \
  --questions {DRIVE_ROOT}/data/questions.json \
  --out {DRIVE_ROOT}/data/densecap_candidates.json

## 7. Build the final training manifest

In [ ]:
!python scripts/build_manifest.py \
  --questions {DRIVE_ROOT}/data/questions.json \
  --features_path {FEATURES_PATH} \
  --candidates {DRIVE_ROOT}/data/densecap_candidates.json \
  --out {DRIVE_ROOT}/data/manifest.json

## 8. Train

Paper-given: batch size 64, 128 epochs (VQA) / 64 epochs (Visual7W) -- `configs/default.yaml` defaults to 128; pass `--epochs 64` for Visual7W. All hyperparameters the paper never specifies (learning rate, hidden sizes, etc.) live in that same config file with inline comments -- edit there, not here, if you want to sweep them.

In [ ]:
epochs_arg = "" if DATASET == "vqa" else "--epochs 64"
!python -m src.train \
  --config configs/default.yaml \
  --manifest {DRIVE_ROOT}/data/manifest.json \
  --features {FEATURES_PATH} \
  --glove {DRIVE_ROOT}/data/glove.840B.300d.txt \
  --out_dir {DRIVE_ROOT}/checkpoints \
  {epochs_arg}

## 9. Generate + evaluate

Reproduces the paper's Fig. 3 precision/recall sweep over N=1..6 generated questions per image (`eval/evaluate.py::sweep_num_questions`). Fill in a checkpoint path and a held-out slice of the manifest before running.

In [ ]:
import json, torch, numpy as np
from tqdm import tqdm
from src.vocab import Vocab
from src.embeddings import build_embedding_matrix
from src.model import GroundedVQGModel
from src.bigram_lm import KneserNeyBigram
from src.dataset import tokenize
from src.generate import generate_questions
from eval.evaluate import sweep_num_questions, group_references_by_image

# checkpoint_best.pt (lowest val_loss during training) rather than a fixed final-epoch
# checkpoint -- see the train/val-split discussion: the model can overfit well before
# reaching the last epoch, and checkpoint_best.pt already tracks whichever epoch
# generalized best, no need to hand-pick an epoch number.
CKPT_PATH = f"{DRIVE_ROOT}/checkpoints/checkpoint_best.pt"
ckpt = torch.load(CKPT_PATH, map_location="cpu")
vocab = Vocab(); vocab.idx2word = ckpt["vocab"]; vocab.word2idx = {w: i for i, w in enumerate(vocab.idx2word)}

embedding = build_embedding_matrix(vocab, dim=300)  # shapes only; real weights load via state_dict below
model = GroundedVQGModel(embedding, vocab_size=len(vocab),
                          type_hidden=ckpt["cfg"]["type_selector_hidden"],
                          decoder_hidden=ckpt["cfg"]["decoder_hidden"])
model.load_state_dict(ckpt["model"])
model.eval()

with open(f"{DRIVE_ROOT}/data/manifest.json") as f:
    manifest = json.load(f)

# Filtered to the "val" split specifically -- previously this took the first 500
# records of the *whole* manifest (train+val mixed), which could be dominated by
# records the model was actually trained on. That undercuts the entire point of
# holding out a val split in the first place: these Fig. 3 numbers need to come from
# genuinely unseen data. 2000 is a size/runtime compromise (recall_scores' pairwise
# reference x candidate comparisons scale with sample size) -- raise it for a more
# robust final number, at the cost of a longer run.
val_records = [r for r in manifest if r.get("split", "train") == "val"]
print(f"{len(val_records)} val records available")
eval_records = val_records[:2000]

# Loaded once here rather than per-record -- same consolidated-features reasoning as
# src/dataset.py (one .npy per image, re-read every epoch, was the cause of the hang
# during training; this eval loop is one-shot so it's less severe, but there's no
# reason to reintroduce per-file Drive reads here either).
features_npz = np.load(FEATURES_PATH)
features = {k: features_npz[k] for k in features_npz.files}

# Fit on train split only, matching src/train.py's vocab/IDF construction -- the
# bigram LM is part of inference-time decoding, not something val data should shape.
train_questions = [tokenize(r["question"]) for r in manifest if r.get("split", "train") == "train"]
bigram_lm = KneserNeyBigram(discount=0.75).fit(train_questions)

# generate_questions' decode loop calls bigram_lm.prob_vector() per step -- cached per
# prev_word internally, so this loop is now dramatically faster than it was (a real
# bug: it used to recompute the bigram distribution over the whole vocabulary from
# scratch on every single decode step, with no caching at all -- billions of redundant
# Python-level calls at this sample size). tqdm here so a long run is visibly
# progressing instead of silent, same reasoning as everywhere else in this notebook.
generated_pool, references = {}, {}
for r in tqdm(eval_records, desc="generating"):
    image_id = str(r["image_id"])
    feat = torch.from_numpy(features[image_id]).float()
    qs = generate_questions(model, vocab, bigram_lm, feat, r["candidates"],
                             num_questions=6, beta=ckpt["cfg"]["bigram_beta"])
    generated_pool.setdefault(image_id, []).extend(qs)
    references.setdefault(image_id, []).append(r["question"])

results = sweep_num_questions(references, generated_pool, max_n=6)
for n, scores in results.items():
    print(n, scores)

## 10. Try it on your own picture

Upload any image and see what questions your trained model generates for it -- not
part of the paper's own evaluation, just for trying the pipeline out interactively.
Re-run this cell as many times as you want for different pictures; the model/vocab/
bigram LM load once and get reused (loading them again is the slow part, not
generation itself).

In [ ]:
from google.colab import files
import json, os, torch

# Cwd can drift across a long session -- DenseCap's own cells %cd into
# /content/densecap-pytorch and back, and if one of those didn't complete cleanly (or
# got skipped), the "cd back" never runs, leaving this cell's files.upload() saving
# into the wrong directory and os.path.abspath() resolving to a path that doesn't
# exist. Resetting it explicitly here makes this cell self-contained regardless of
# what happened earlier in the session.
os.chdir("/content/grounded-vqg-reproduction")
from src.vocab import Vocab
from src.embeddings import build_embedding_matrix
from src.model import GroundedVQGModel
from src.bigram_lm import KneserNeyBigram
from src.dataset import tokenize
from src.generate import generate_questions
from scripts.generate_for_image import extract_single_image_feature
from src.object_detector import detect_objects, detect_objects_as_candidates, detect_relations_as_candidates
from src.blip_captioner import caption_whole_image, caption_regions_as_candidates

# Loads once per session and reuses -- if you already ran cell 9, this reuses that
# model/vocab/bigram_lm instead of reloading from scratch.
if "model" not in dir() or "bigram_lm" not in dir():
    print("Loading model + vocab + bigram LM (first time this session)...")
    ckpt_path = f"{DRIVE_ROOT}/checkpoints/checkpoint_best.pt"
    ckpt = torch.load(ckpt_path, map_location="cpu")
    vocab = Vocab(); vocab.idx2word = ckpt["vocab"]; vocab.word2idx = {w: i for i, w in enumerate(vocab.idx2word)}
    embedding = build_embedding_matrix(vocab, dim=300)
    model = GroundedVQGModel(embedding, vocab_size=len(vocab),
                              type_hidden=ckpt["cfg"]["type_selector_hidden"],
                              decoder_hidden=ckpt["cfg"]["decoder_hidden"])
    model.load_state_dict(ckpt["model"])
    model.eval()

    with open(f"{DRIVE_ROOT}/data/manifest.json") as f:
        _manifest = json.load(f)
    train_questions = [tokenize(r["question"]) for r in _manifest if r.get("split", "train") == "train"]
    bigram_lm = KneserNeyBigram(discount=0.75).fit(train_questions)

print("Choose an image file to upload:")
uploaded = files.upload()
# files.upload() saves into the notebook's CURRENT working directory, not /content/
# specifically -- since cell 1 leaves cwd at /content/grounded-vqg-reproduction, a
# hardcoded "/content/" prefix pointed at the wrong path entirely. os.path.abspath()
# resolves the bare filename against whatever cwd actually is, correctly either way.
image_path = os.path.abspath(list(uploaded.keys())[0])
print(f"Uploaded to: {image_path}")

print("Extracting VGG-16 feature...")
# VGG-16 stays exactly as the paper specifies (Sec 3.2: "the image features from
# VGG-16") -- this is a DIFFERENT role than caption generation (a numeric feature
# vector, not text), so replacing DenseCap below doesn't touch this at all.
image_feat = extract_single_image_feature(image_path)

# Pretrained (COCO-trained, no training/API key needed) object detector used both for
# region proposals (below, BLIP captions each detected box) and as a reliability
# cross-check -- see src/object_detector.py. This is what confirmed several earlier
# hallucinations (bird, mouse, cake, vase, scissors, banana) are literally COCO class
# names the decoder defaults to regardless of caption/image -- object_suppression_bias
# actively penalizes those specific words unless this detector actually confirms them.
print("Running object detector for regions + a reliability cross-check...")
detected = detect_objects(image_path, confidence_threshold=0.5)
print(f"Detector confirms: {detected or '(nothing above threshold)'}")

# BLIP (Salesforce, ~247M params) replaces DenseCap's role entirely -- DenseCap was
# trained from scratch (~0.09 mAP) and repeatedly introduced wrong/irrelevant content
# across real test images ("plate", "food", "hand" -- none ever confirmed by the
# detector). BLIP ships pretrained and produces far more reliable natural-language
# captions. First call downloads its weights (~1GB) -- see src/blip_captioner.py and
# the weight/fidelity discussion for why this doesn't touch paper fidelity (DenseCap
# was always an external, swappable input source, same as this) but IS a real added
# weight cost. BLIP captions one crop at a time, so caption_regions_as_candidates runs
# it over the same detected boxes above to reconstruct DenseCap's multi-candidate
# format; caption_whole_image adds one more high-trust overview caption.
print("Running BLIP over each detected region (first call downloads ~1GB, be patient)...")
blip_region_candidates = caption_regions_as_candidates(image_path, confidence_threshold=0.3)
blip_whole_image_candidate = caption_whole_image(image_path)
print(f"BLIP captions ({len(blip_region_candidates) + 1} total):")
for c in [blip_whole_image_candidate] + blip_region_candidates:
    print(f"   {c['confidence']:.2f}  {c['caption']!r}")

# Detector-derived candidates (color/size/position + geometric relations) still add
# their own zero-hallucination-risk, purely deterministic content alongside BLIP's
# richer natural-language captions.
detector_candidates = detect_objects_as_candidates(image_path, confidence_threshold=0.5)
relation_candidates = detect_relations_as_candidates(image_path, confidence_threshold=0.5)
candidates = [blip_whole_image_candidate] + blip_region_candidates + detector_candidates + relation_candidates
print(f"Using {len(candidates)} total candidates (BLIP + detector + relations).")

# beam_width=5, dedup_captions=True, distinct_types=True, lexical_bias=4.0,
# object_suppression_bias=4.0 -- all inference-time-only knobs (cell 9's validated
# Fig. 3 numbers keep the defaults, unaffected). DenseCap often proposes several
# overlapping boxes describing the same thing (e.g. "the hand of a person" showing up
# 6 times among 20 candidates for one image) -- dedup_captions collapses those before
# sampling. The type-selector's distribution is often skewed heavily toward "what"
# (~85-90% in practice), so independently sampling a type per question tends to
# collapse most/all of the 6 questions onto the same type -- distinct_types forces the
# 6 draws to cover different types instead. lexical_bias adds a LOG-space bonus to
# vocab words that literally appear in the sampled caption at every decode step (each
# +1 is ~2.7x more probability weight); object_suppression_bias subtracts the same
# kind of bonus from COCO-class words the detector above did NOT confirm. Lowered from
# 8.0 to 4.0 here now that candidate pooling does some of the work -- raise either (up
# to 15-20) if still hallucinating, lower them if questions read as word salad.
# no_repeat_content_words blocks a content word from being generated twice within the
# same question -- without it, lexical_bias/object_suppression_bias apply the same flat
# bonus/penalty at every step with no memory of what's already been generated, which
# produced degenerate loops like "white white white white ..." in practice.
questions = generate_questions(model, vocab, bigram_lm, image_feat, candidates,
                                num_questions=6, beta=ckpt["cfg"]["bigram_beta"],
                                beam_width=5, dedup_captions=True, distinct_types=True,
                                lexical_bias=4.0, object_suppression_bias=4.0,
                                detected_objects=detected, no_repeat_content_words=True)

from PIL import Image
import matplotlib.pyplot as plt
img = Image.open(image_path).convert("RGB")
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

print("Generated questions:")
for q in questions:
    print(" -", q)

## 11. EXPERIMENTAL: attention-augmented decoder

Not part of the paper's architecture -- Sec 3.2's decoder only ever sees one
fixed 300-d joint feature vector, computed once before generation starts, with
no way to look back at the caption's actual words while writing the question.
On an undertrained checkpoint (what this reproduction's compute/data budget
produces), the decoder tends to fall back on frequent, caption-independent
words instead of using that compressed signal -- see cell 10's demo outputs
above.

This adds Bahdanau-style attention so the decoder can look back at individual
caption words at every decoding step -- the standard fix for exactly this kind
of bottleneck. It's a real architecture change, so it lives entirely in
`src/attention_experiment.py` and `scripts/train_attention_experiment.py`,
completely separate from the paper-faithful model/training run in cells 1-10
(nothing above this point is touched). Attention here is only over the
caption's own tokens (a handful of words), not image regions/pixels -- the
expensive kind of attention the paper's own "lightweight" framing avoids, so
it stays in the same spirit. The next cell prints an exact parameter-count
comparison against the paper-faithful model before you commit to training it.

This is a proof-of-concept comparison, not a full 128-epoch commitment -- run
it for far fewer epochs (e.g. 15-20) first to see whether attention actually
helps before spending more Colab GPU time.

In [ ]:
# Proof-of-concept run -- 15 epochs by default (vs. the paper-faithful model's
# 128) since the point here is just to see whether attention helps at all before
# committing more GPU time. Raise ATTENTION_EPOCHS if the trend looks promising.
ATTENTION_EPOCHS = 15
ATTENTION_OUT_DIR = f"{DRIVE_ROOT}/checkpoints_attention_experiment"

!python -m scripts.train_attention_experiment \
  --config configs/default.yaml \
  --manifest {DRIVE_ROOT}/data/manifest.json \
  --features {FEATURES_PATH} \
  --glove {DRIVE_ROOT}/data/glove.840B.300d.txt \
  --out_dir {ATTENTION_OUT_DIR} \
  --epochs {ATTENTION_EPOCHS}

In [ ]:
# Compares the attention-experiment checkpoint against checkpoint_best.pt
# (paper-faithful model) on the SAME image/candidates already in memory from
# cell 10 -- no re-upload needed. Uses greedy decoding only (no beam search/
# dedup/distinct-types yet -- see src/attention_experiment.py's docstring).
import torch
from src.attention_experiment import AttentionGroundedVQGModel, generate_questions_attention
from src.vocab import Vocab
from src.embeddings import build_embedding_matrix

attn_ckpt_path = f"{ATTENTION_OUT_DIR}/checkpoint_best.pt"
attn_ckpt = torch.load(attn_ckpt_path, map_location="cpu")
attn_vocab = Vocab()
attn_vocab.idx2word = attn_ckpt["vocab"]
attn_vocab.word2idx = {w: i for i, w in enumerate(attn_vocab.idx2word)}
attn_embedding = build_embedding_matrix(attn_vocab, dim=300)
attn_model = AttentionGroundedVQGModel(attn_embedding, vocab_size=len(attn_vocab),
                                        type_hidden=attn_ckpt["cfg"]["type_selector_hidden"],
                                        decoder_hidden=attn_ckpt["cfg"]["decoder_hidden"])
attn_model.load_state_dict(attn_ckpt["model"])
attn_model.eval()

print(f"attention-experiment epoch {attn_ckpt.get('epoch', '?')}: "
      f"train_loss={attn_ckpt['train_loss']:.4f} val_loss={attn_ckpt['val_loss']:.4f}")

questions_attn = generate_questions_attention(attn_model, attn_vocab, image_feat, candidates,
                                               num_questions=6)
print("Generated questions (attention experiment):")
for q in questions_attn:
    print(" -", q)